# Gradient Extraction & Storing

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np

#########################################
# 1. Define a simple CNN with 3 conv layers
#########################################
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # conv1: from 4x4 input → output remains 4x4
        self.conv1 = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1, bias=False)
        # conv2: from 4x4 → 2x2 (stride=2)
        self.conv2 = nn.Conv2d(1, 1, kernel_size=3, stride=2, padding=1, bias=False)
        # conv3: from 2x2 → 1x1 (stride=2)
        self.conv3 = nn.Conv2d(1, 1, kernel_size=3, stride=1, padding=1, bias=False)
        self.fcn = nn.Linear(4, 10, bias=False)  # not used for interaction

    def forward(self, x):
        
        a1 = (self.conv1(x))    # shape: (1,1,4,4)
        a2 = (self.conv2(a1))     # shape: (1,1,2,2)
        a3 = (self.conv3(a2))     # shape: (1,1,1,1)
        return a3

model = SimpleCNN()
criterion = nn.CrossEntropyLoss(reduction="none")
optimizer = optim.SGD(model.parameters(), lr=0.01)


#########################################
# 2. Register hooks to capture gradients
#########################################
activation_gradients = {}
gradient_flows = {}
def get_activation_grad(name, connet2name=None):
    def hook(module, grad_input, grad_output):
        if name is not None:
            # Lấy grad_out: shape [N, C_out, H_out, W_out]
            grad_out = grad_output[0].detach()
            N, C_out, H_out, W_out = grad_out.shape
            
            # Lấy weight của module: shape [C_out, C_in, kH, kW]
            weight = module.weight  
            C_out_w, C_in, kH, kW = weight.shape
            assert C_out == C_out_w, "Mismatch in output channels."

            # Chuyển weight thành dạng ma trận: [C_out, C_in*kH*kW]
            weight_reshaped = weight.view(C_out, -1)
            
            # Reshape grad_out thành [N, C_out, Len_out] với Len_out = H_out * W_out
            Len_out = H_out * W_out
            grad_out_reshaped = grad_out.view(N, C_out, Len_out)

            # Tính grad_input_cols: [N, C_in*kH*kW, Len_out]
            grad_input_cols = torch.matmul(weight_reshaped.t(), grad_out_reshaped)
            
            # Giả sử batch size N=1
            grad_input_cols = grad_input_cols[0]  # [C_in*kH*kW, Len_out]

            # Lấy kích thước input từ grad_input[0]: [N, C_in, H_in, W_in]
            H_in, W_in = grad_input[0].shape[2:]
            Len_in = H_in * W_in

            # Xây dựng ánh xạ từ các patch đến các vị trí trên input:
            # Tạo tensor chứa các chỉ số của các ô input, shape: [1, 1, H_in, W_in]
            input_indices = torch.arange(Len_in, device=grad_out.device).view(1, 1, H_in, W_in).float()+1
            # Sử dụng F.unfold để lấy ma trận ánh xạ, shape: [C_in*kH*kW, Len_out]
            idx_map = (F.unfold(input_indices, kernel_size=module.kernel_size, 
                                dilation=module.dilation, padding=module.padding, stride=module.stride)[0])

            # Khởi tạo gradient_flows với kích thước (Len_in, Len_out)
            gradient_flow = torch.zeros(Len_in+1, Len_out, device=grad_out.device)
            # Sử dụng scatter_add_ để cộng các giá trị từ grad_input_cols vào gradient_flows
            # Cho mỗi phần tử tại vị trí (p, j) trong grad_input_cols, ta cộng vào gradient_flows tại (idx_map[p,j], j)
            gradient_flow.scatter_add_(0, idx_map.long(), grad_input_cols)
            gradient_flow = (gradient_flow[1:,:])
            # print((grad_input[0]).cpu().numpy().reshape(-1))
            # print(gradient_flow.cpu().numpy().sum(axis=-1,keepdims=False))
            assert np.abs(gradient_flow.cpu().numpy().sum(axis=-1,keepdims=False)-(grad_input[0]).cpu().numpy().reshape(-1)).sum() < 1e-7, "Mismatch in gradient values."
            gradient_flow = torch.abs(gradient_flow)
            gradient_flow[gradient_flow>1e-5] = 1.
            gradient_flow[gradient_flow<=1e-5] = .99

            # Lưu kết quả vào activation_gradients
            gradient_flows[(name, connet2name)] = gradient_flow.cpu().numpy()  # kích thước: (Len_in, Len_out)
            
            activation_gradients[name] = (gradient_flows[(name, connet2name)]).sum(axis=-1,keepdims=False).reshape(H_in, W_in)
            activation_gradients[connet2name] = (gradient_flows[(name, connet2name)]).sum(axis=0,keepdims=False).reshape(H_out, W_out)
    return hook

model.conv1.register_backward_hook(get_activation_grad(None, "conv1"))
model.conv2.register_backward_hook(get_activation_grad("conv1", "conv2"))
model.conv3.register_backward_hook(get_activation_grad("conv2", "conv3"))

#########################################
# 3. Run forward/backward on a random input
#########################################
input_tensor = torch.randn(1, 1, 4, 4)
target = torch.randn(1, 1, 4, 4).long()
optimizer.zero_grad()
a3 = model(input_tensor)
loss = (a3.reshape(1,-1) - target[0,0,2,2].reshape(1,-1)).mean()  # use conv3 output for loss
loss.backward()

Save gradient flow information, includes: 
- activation_gradients: dict of aggregated weights of nodes at each layer. 
- gradient_flows: dict of values of gradient flows between adjacent layers.

In [1]:
import pickle
flow_info = {"input_representation": input_tensor[0,0,:,:].cpu().numpy(), 
             "activation_gradients": activation_gradients, 
             "gradient_flows": gradient_flows}
with open('flow_info.pkl', 'wb') as f:
    pickle.dump(flow_info, f)
np.save('target_representation.npy', target[0,0,:,:].cpu().numpy())

NameError: name 'input_tensor' is not defined

# Rendering the Visualization

In [1]:
import numpy as np
from torchvision import transforms
import plotly.graph_objs as go
import ipywidgets as widgets
from IPython.display import display
from matplotlib import cm as cm
from matplotlib import colors as mcolors
import pickle

#########################################
# 1. Load data: gradients, input and target representations
#########################################
with open('flow_info.pkl', 'rb') as f:
    flow_info = pickle.load(f)
input_representation = flow_info["input_representation"]
activation_gradients = flow_info["activation_gradients"]
gradient_flows = flow_info["gradient_flows"]

# target_representation bây giờ được load từ file numpy
target_representation = np.load('target_representation.npy')

In [2]:
import numpy as np
from torchvision import transforms
import plotly.graph_objs as go
import ipywidgets as widgets
from IPython.display import display
from matplotlib import cm as cm
from matplotlib import colors as mcolors
import pickle

# Tạo bản sao chỉnh sửa của input (để người dùng tương tác: toggle giữa 0 và 1)
input_representation_edit = input_representation.copy()

#########################################
# 2. Process gradients: obtain heatmaps and flattened vectors
#########################################
grad_vecs = dict()
node_layers, selectable_layers = [], []
for key, value in activation_gradients.items():
    node_layers.append(key)
    if type(key) is not tuple:
        grad_vecs[key] = value.reshape(-1)
for key, value in gradient_flows.items():
    node_layers.append(key[0])
    node_layers.append(key[1])
    if key[0] not in grad_vecs:
        grad_vecs[key[0]] = value.sum(axis=-1, keepdims=True)
    selectable_layers.append(key[0])
    if key[1] not in grad_vecs:
        grad_vecs[key[1]] = value.sum(axis=0, keepdims=True)
node_layers = list(sorted(set(node_layers)))
selectable_layers = list(sorted(set(selectable_layers)))

#########################################
# 3. Compute flow matrices via outer product, with thresholding
#########################################
def compute_flow(gradient_flows, grad_vecs):
    flow_matrices = dict()
    for source, source_grad in grad_vecs.items():
        for target, target_grad in grad_vecs.items():
            key = (source, target)
            if key in gradient_flows:
                value = gradient_flows[key]
                if value is None:
                    value = source_grad.reshape(-1, 1) / target_grad.reshape(1, -1)
                    value[np.isnan(value)] = 0
                    value = value / np.sum(value, axis=-1, keepdims=True)
                    value[np.isnan(value)] = 0
                flow_matrices[key] = value * 1.
    return flow_matrices

flow_matrices = compute_flow(gradient_flows, grad_vecs)

# Determine dynamic threshold slider limits:
flow_slider_min, flow_slider_max = None, None
for key, value in flow_matrices.items():
    if (flow_slider_min is None) or (value.min() < flow_slider_min):
        flow_slider_min = value.min()
    if (flow_slider_max is None) or (value.max() > flow_slider_max):
        flow_slider_max = value.max()

#########################################
# 4. Build Sankey data for layers.
#########################################
global_index = {}  # Mapping: key = (layer, (row, col))
node_labels = []
current_index = 0

# Sử dụng colormaps khác nhau cho các node:
colors = dict()
color_coin = 0
node_colors = list()
for key in node_layers:
    value = grad_vecs[key]
    if color_coin % 2:
        color_list = [mcolors.to_hex(cm.viridis(i/len(value))) for i in range(len(value))]
    else:
        color_list = [mcolors.to_hex(cm.plasma(i/len(value))) for i in range(len(value))]
    colors[key] = color_list
    color_coin += 1
    node_colors += color_list
    
# Gán vị trí cố định cho các node dựa trên thứ tự của heatmap.
node_orders = dict()
node_x, node_y = [], []
for i, label in enumerate(node_layers):
    node_orders[label] = [(r, c) for r in range(activation_gradients[label].shape[-2])
                          for c in range(activation_gradients[label].shape[-1])]
    for j, (r, c) in enumerate(node_orders[label]):
        node_labels.append(f"{label}_{r},{c}")
        node_x.append((float(i) + 0.01) / (len(node_layers) - 0.55))
        node_y.append((float(j) + 0.5) / (len(node_orders[label])))
        global_index[(label, (r, c))] = current_index
        current_index += 1

new_sources, new_targets, new_values = [], [], []
for key, value in flow_matrices.items():
    source, target = key
    source_order = node_orders[source]
    target_order = node_orders[target]
    for i, (r1, c1) in enumerate(source_order):
        for j, (r2, c2) in enumerate(target_order):
            v = value[i, j]
            if v > 0:
                new_sources.append(global_index[(source, (r1, c1))])
                new_targets.append(global_index[(target, (r2, c2))])
                new_values.append(v)

#########################################
# 5. Create functions to generate Plotly figures (Sankey & Heatmaps)
#########################################
def create_sankey(selected_layer, selected_index, threshold):
    new_node_colors = []
    new_link_colors = []
    for key in node_layers:
        new_node_color = colors[key]
        if selected_layer == key:
            new_node_color = [("red" if i == selected_index else new_node_color[i])
                              for i in range(len(grad_vecs[selected_layer]))]
        new_node_colors += new_node_color
    for vi, src in enumerate(new_sources):
        new_link_colors.append("red" if (src == global_index[(selected_layer, node_orders[selected_layer][selected_index])])
                               and (float(new_values[vi]) > threshold)
                               else f"rgba({128*int(float(new_values[vi])>threshold)},"
                                    f"{128*int(float(new_values[vi])>threshold)},"
                                    f"{128*int(float(new_values[vi])>threshold)},0.1)")
    sankey_fig = go.FigureWidget(data=[go.Sankey(
        arrangement='fixed',
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color='black', width=0.5),
            label=node_labels,
            color=new_node_colors,
            x=node_x,
            y=node_y
        ),
        link=dict(
            source=new_sources,
            target=new_targets,
            value=new_values,
            color=new_link_colors
        )
    )])
    sankey_fig.update_layout(title_text='Gradient Flow', font_size=10)
    return sankey_fig

def create_heatmap_conv1(selected_index, selected_layer):
    shapes = []
    grid_size = activation_gradients[selected_layer].shape
    if selected_index is not None:
        row, col = divmod(selected_index, grid_size[-1])
        shapes.append(dict(
            type='circle', xref='x', yref='y',
            x0=col - 0.5, y0=grid_size[0]-1 - row - 0.5,
            x1=col + 0.5, y1=grid_size[0]-1 - row + 0.5,
            line=dict(color='red', width=3)
        ))
    heatmap1_fig = go.FigureWidget(data=go.Heatmap(
        z=(activation_gradients[selected_layer])[::-1, :],
        colorscale='Viridis',
        zmin=(activation_gradients[selected_layer]).min(),
        zmax=(activation_gradients[selected_layer]).max(),
        colorbar=dict(title='',
                      len=0.5,
                      x=1.02,
                      xanchor='left',
                      thickness=10)
    ))
    heatmap1_fig.update_layout(
        margin=dict(l=10, r=10, t=10, b=40),
        width=300, height=300,
        xaxis=dict(dtick=1),
        yaxis=dict(dtick=1),
        shapes=shapes,
        annotations=[dict(
            text="Selected-Layer Gradient",
            x=0.5, y=-0.15,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10)
        )]
    )
    return heatmap1_fig

def create_heatmap_conv2(selected_index, selected_layer, threshold):
    shapes = []
    connected_layer = selected_layer
    for key in gradient_flows.keys():
        if key[0] == selected_layer:
            connected_layer = key[1]
    grid_size = activation_gradients[connected_layer].shape
    flow_matrix = ((flow_matrices[(selected_layer, connected_layer)] > threshold)[selected_index, :]).reshape(*grid_size)
    for r in range(grid_size[0]):
        for c in range(grid_size[1]):
            if flow_matrix[r, c] > 0.:
                shapes.append(dict(
                    type='circle', xref='x', yref='y',
                    x0=c - 0.5, y0=grid_size[0]-1 - r - 0.5,
                    x1=c + 0.5, y1=grid_size[0]-1 - r + 0.5,
                    line=dict(color='red', width=3)
                ))
    heatmap2_fig = go.FigureWidget(data=go.Heatmap(
        z=(activation_gradients[connected_layer])[::-1, :],
        colorscale='Plasma',
        zmin=(activation_gradients[connected_layer]).min(),
        zmax=(activation_gradients[connected_layer]).max(),
        colorbar=dict(title='',
                      len=0.5,
                      x=1.02,
                      xanchor='left',
                      thickness=10)
    ))
    heatmap2_fig.update_layout(
        margin=dict(l=10, r=10, t=10, b=40),
        width=300, height=300,
        xaxis=dict(dtick=1),
        yaxis=dict(dtick=1),
        shapes=shapes,
        annotations=[dict(
            text="Output-Layer Gradient",
            x=0.5, y=-0.15,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10)
        )]
    )
    return heatmap2_fig

def create_heatmap_target(selected_index):
    shapes = []
    grid_size = target_representation.shape
    if selected_index is not None:
        row, col = divmod(selected_index, grid_size[-1])
        shapes.append(dict(
            type='circle', xref='x', yref='y',
            x0=col - 0.5, y0=grid_size[0]-1 - row - 0.5,
            x1=col + 0.5, y1=grid_size[0]-1 - row + 0.5,
            line=dict(color='red', width=3)
        ))
    target_fig = go.FigureWidget(data=go.Heatmap(
        z=target_representation[::-1, :],
        colorscale='Viridis',
        zmin=target_representation.min(),
        zmax=target_representation.max(),
        colorbar=dict(title='',
                      len=0.5,
                      x=1.02,
                      xanchor='left',
                      thickness=10)
    ))
    target_fig.update_layout(
        margin=dict(l=10, r=10, t=10, b=40),
        width=300, height=300,
        xaxis=dict(dtick=1),
        yaxis=dict(dtick=1),
        shapes=shapes,
        annotations=[dict(
            text="Target Representation",
            x=0.5, y=-0.15,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10)
        )]
    )
    return target_fig

def create_all_heatmaps(selected_index, selected_layer, threshold):
    heatmap1 = create_heatmap_conv1(selected_index, selected_layer)
    heatmap2 = create_heatmap_conv2(selected_index, selected_layer, threshold)
    target_fig = create_heatmap_target(selected_index)
    return heatmap1, heatmap2, target_fig

def create_heatmap_input():
    # Hiển thị input_representation_edit theo dạng heatmap với colormap nhị phân (0 = white, 1 = black)
    input_fig = go.FigureWidget(data=go.Heatmap(
        z=input_representation_edit[::-1, :],
        colorscale=[[0, 'white'], [1, 'black']],
        zmin=0,
        zmax=1,
        colorbar=dict(title='',
                      len=0.5,
                      x=1.02,
                      xanchor='left',
                      thickness=10)
    ))
    input_fig.update_layout(
        margin=dict(l=10, r=10, t=10, b=40),
        width=300, height=300,
        xaxis=dict(dtick=1),
        yaxis=dict(dtick=1),
        annotations=[dict(
            text="Input Representation",
            x=0.5, y=-0.15,
            xref="paper", yref="paper",
            showarrow=False,
            font=dict(size=10)
        )]
    )
    return input_fig

# Callback xử lý sự kiện click trên heatmap Input.
def input_on_click(trace, points, state):
    if points.point_inds:
        rows, cols = input_representation_edit.shape
        r_display = (points.point_inds[0])[0]
        c = (points.point_inds[0])[1]
        r = rows - 1 - r_display
        # Toggle giá trị: nếu > 0.5 thì set về 0, ngược lại 1
        input_representation_edit[r, c] = 0.0 if input_representation_edit[r, c] > 0.5 else 1.0
        input_widget.data[0].z = input_representation_edit[::-1, :]

#########################################
# 6. IPyWidgets for Interaction and Edit/Save Button
#########################################
layer_dropdown = widgets.Dropdown(
    options=selectable_layers,
    value=selectable_layers[0],
    description='Select Layer:'
)

node_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=(np.prod(activation_gradients[layer_dropdown.value].shape[-2:])) - 1,
    step=1,
    description='Node Index:'
)

threshold_slider = widgets.FloatSlider(
    value=(flow_slider_max - flow_slider_min)/2,
    min=flow_slider_min,
    max=flow_slider_max,
    step=(flow_slider_max - flow_slider_min) / 100,
    description='Threshold:'
)

def update_slider_range(selected_layer):
    node_slider.min = 0
    node_slider.max = (np.prod(activation_gradients[selected_layer].shape[-2:])) - 1
    if node_slider.value > node_slider.max:
        node_slider.value = 0

# Nút Edit/Save: Ban đầu hiển thị "Edit". Khi bấm, chuyển sang Input Editing mode (nút đổi thành "Save").
# Khi bấm "Save", lưu lại input_representation_edit và thực hiện (TODO) gọi mạng nơ-ron.
input_editing_mode = False  # False: Gradient debug mode; True: Input Editing mode
edit_save_button = widgets.Button(description="Edit", button_style='primary')

def on_edit_save_button_clicked(b):
    global input_editing_mode
    if not input_editing_mode:
        # Chuyển sang Input Editing mode
        input_editing_mode = True
        edit_save_button.description = "Save"
        # Cho phép tương tác với input heatmap
        input_widget.data[0].on_click(input_on_click)
    else:
        # Người dùng bấm Save: lưu trạng thái hiện tại của input
        input_editing_mode = False
        edit_save_button.description = "Edit"
        # Vô hiệu callback để ngăn chỉnh sửa thêm
        input_widget.data[0].on_click(None)
        # TODO: call neural network for getting new gradient information.
        # Sau khi có thông tin gradient mới, cập nhật các biến activation_gradients, gradient_flows, v.v...
        
edit_save_button.on_click(on_edit_save_button_clicked)

# Khởi tạo các FigureWidget toàn cục để update thay vì tạo mới mỗi lần
sankey_widget = create_sankey(layer_dropdown.value, node_slider.value, threshold_slider.value)
heatmap_conv1_widget, heatmap_conv2_widget, target_widget = create_all_heatmaps(node_slider.value, layer_dropdown.value, threshold_slider.value)
input_widget = create_heatmap_input()
# Ban đầu, không cho phép chỉnh sửa input (đã ở Gradient debug mode)
input_widget.data[0].on_click(None)

# Tạo container cho các figure
output_sankey = widgets.VBox([sankey_widget])
output_heatmaps = widgets.HBox([heatmap_conv1_widget, heatmap_conv2_widget, target_widget])
output_input = widgets.VBox([edit_save_button, input_widget])

def update_figures(selected_layer, selected_index, threshold):
    # Cập nhật Sankey Diagram
    new_sankey = create_sankey(selected_layer, selected_index, threshold)
    sankey_widget.data[0].node.label = new_sankey.data[0].node.label
    sankey_widget.data[0].node.color = new_sankey.data[0].node.color
    sankey_widget.data[0].node.x = new_sankey.data[0].node.x
    sankey_widget.data[0].node.y = new_sankey.data[0].node.y
    sankey_widget.data[0].link.source = new_sankey.data[0].link.source
    sankey_widget.data[0].link.target = new_sankey.data[0].link.target
    sankey_widget.data[0].link.value = new_sankey.data[0].link.value
    sankey_widget.data[0].link.color = new_sankey.data[0].link.color

    # Cập nhật 3 Heatmaps (conv1, conv2, target)
    new_heatmap1, new_heatmap2, new_target = create_all_heatmaps(selected_index, selected_layer, threshold)
    
    heatmap_conv1_widget.data[0].z = new_heatmap1.data[0].z
    heatmap_conv1_widget.layout.title.text = new_heatmap1.layout.title.text
    heatmap_conv1_widget.layout.shapes = new_heatmap1.layout.shapes
    heatmap_conv1_widget.layout.annotations = new_heatmap1.layout.annotations

    heatmap_conv2_widget.data[0].z = new_heatmap2.data[0].z
    heatmap_conv2_widget.layout.title.text = new_heatmap2.layout.title.text
    heatmap_conv2_widget.layout.shapes = new_heatmap2.layout.shapes
    heatmap_conv2_widget.layout.annotations = new_heatmap2.layout.annotations

    target_widget.data[0].z = new_target.data[0].z
    target_widget.layout.title.text = new_target.layout.title.text
    target_widget.layout.shapes = new_target.layout.shapes
    target_widget.layout.annotations = new_target.layout.annotations

def on_interaction_change(change):
    layer = layer_dropdown.value
    thresh = threshold_slider.value
    update_slider_range(layer)
    node = node_slider.value
    update_figures(layer, node, thresh)

layer_dropdown.observe(on_interaction_change, names='value')
node_slider.observe(on_interaction_change, names='value')
threshold_slider.observe(on_interaction_change, names='value')

# Sắp xếp giao diện:
# - Hàng đầu tiên: các widget điều khiển (layer, node, threshold)
# - Hàng thứ hai: Sankey Diagram
# - Hàng thứ ba: 3 Heatmaps (conv1, conv2, target)
# - Hàng thứ tư: Nút Edit/Save và Heatmap của Input
ui = widgets.VBox([
    output_sankey,
    widgets.HBox([layer_dropdown, node_slider, threshold_slider]),
    output_heatmaps,
    output_input
])
display(ui)


    'data': [{'arrangement': 'fixed',
              'link': {'col…